# Convert Horstmann CodeCheck items to Catalog v2

This notebook converts `horstmann-java.json` and `horstmann-cpp.json` into one Catalog v2 JSON file per item. Run the verification cell, inspect the output, and set `PUSH_TO_SERVER = True` only when the files are ready to upload.

In [40]:
# Source extraction runs after conversion, in the verification cell below.


In [41]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import requests
from html.parser import HTMLParser
import re

BASE = Path.cwd()
if not (BASE / 'horstmann-java.json').exists():
    BASE = Path('scripts/cay-cpp-java')
OUTPUT = BASE / 'catalogv2-items'
SOURCES = [BASE / 'horstmann-java.json', BASE / 'horstmann-cpp.json']
API_BASE = os.environ.get('CATALOG_V2_API', 'http://adapt2.sis.pitt.edu/next.course-authoring/api')
API_TOKEN_FILE = BASE.parent / 'catalogv2_fixes' / 'api_token.txt'
IMPORT_TIMESTAMP = datetime.now(timezone.utc).isoformat(timespec='milliseconds').replace('+00:00', 'Z')

class CodeParser(HTMLParser):
    def __init__(self):
        super().__init__(); self.parts = []; self.active = False
    def handle_starttag(self, tag, attrs):
        attrs = dict(attrs)
        marker = ' '.join(str(attrs.get(k, '')) for k in ('id', 'name', 'class', 'data-source'))
        if tag in {'textarea', 'pre', 'code'} and re.search(r'source|solution|code|program', marker, re.I):
            self.active = True
    def handle_endtag(self, tag):
        if self.active and tag in {'textarea', 'pre', 'code'}: self.active = False
    def handle_data(self, data):
        if self.active: self.parts.append(data)

def extract_source(url):
    print(f'[fetch] {url}')
    response = requests.get(url, timeout=60)
    print(f"[response] status={response.status_code} bytes={len(response.content)} content_type={response.headers.get('content-type', '')}")
    response.raise_for_status()
    parser = CodeParser(); parser.feed(response.text)
    candidates = [''.join(parser.parts).strip()]
    marker = 'horstmann_codecheck.setup.push('
    marker_pos = response.text.find(marker)
    if marker_pos >= 0:
        try:
            start = response.text.find('{', marker_pos)
            setup, _ = json.JSONDecoder().raw_decode(response.text[start:])
            for file_info in setup.get('requiredFiles', {}).values():
                editors = file_info.get('editors', [])
                candidates.append(''.join(part for part in editors if isinstance(part, str)))
            print(f'[javascript] requiredFiles={len(setup.get("requiredFiles", {}))}')
        except Exception as exc:
            print(f'[javascript] requiredFiles parse failed: {exc}')
    for pattern in (r'\"source_code\"\s*:\s*\"(.*?)\"', r'\"source\"\s*:\s*\"(.*?)\"'):
        candidates.extend(re.findall(pattern, response.text, re.S))
    candidates = [c.replace('\\n', '\n').replace('\\\"', '\"') for c in candidates]
    lengths = [len(c.strip()) for c in candidates]
    print(f'[parse] candidates={len(candidates)} lengths={lengths}')
    source = max(candidates, key=len).strip()
    if not source: raise ValueError('no source-code block found')
    print(f'[success] source_code characters={len(source)}')
    return source

def stable_id(persistent_id):
    # Catalog item IDs are 24 hex characters; hashing keeps reruns idempotent.
    return hashlib.sha256(persistent_id.encode('utf-8')).hexdigest()[:24]

def identity_id(persistent_id):
    return persistent_id.removeprefix('https://codecheck.io/files/')


In [42]:
def convert_item(old):
    pid = old['persistentID']
    language = old.get('programming_language', [''])[0]
    keywords = old.get('keywords', [])
    protocols = old.get('protocol', [])
    urls = old.get('protocol_url', [])
    delivery = [{'protocol': p, 'url': u} for p, u in zip(protocols, urls)]
    return {
        'id': stable_id(pid),
        'user_email': 'moh70@pitt.edu',
        'status': 'public',
        'listed_at': IMPORT_TIMESTAMP,
        'tags': [],
        'identity': {'id': identity_id(pid), 'title': identity_id(pid)[len('horstmann/'):], 'type': 'FreeCodingProblems'},
        'links': {'demo_url': old.get('iframe_url', pid)},
        'attribution': {
            'created_at': IMPORT_TIMESTAMP, 'provider': old.get('platform_name', ''),
            'publisher': old.get('author', [''])[0] if old.get('author') else '',
            'authors': [{'name': a, 'affiliation': old.get('institution', [''])[0] if old.get('institution') else ''} for a in old.get('author', [])]
        },
        'languages': {'content_language': old.get('natural_language', ['en'])[0], 'programming_languages': old.get('programming_language', [])},
        'content': {'prompt': old.get('description', ''), 'source_code': ''},
        'classification': {'topics': [], 'difficulty': '', 'knowledge_components': {'CayHorstmann-keywords': {'concepts': keywords, 'note': 'extracted from the SPLICE.SLCItem.keywords field'}}},
        'pedagogy': {'learning_objectives': [], 'instructional_role': '', 'prerequisites': {'topics': [], 'concepts': [], 'item_ids': []}},
        'interaction': {'interaction_type': ''},
        'delivery': delivery,
        'rights': {'license': 'CC BY-NC-ND 4.0', 'license_url': old.get('license', ''), 'usage_notes': ''},
        'uses': [], 'created_at': IMPORT_TIMESTAMP
    }

OUTPUT.mkdir(exist_ok=True)
converted = []
failures = []
for source in SOURCES:
    converted.extend(json.loads(source.read_text()))
for old in converted:
    item = convert_item(old)
    output_path = OUTPUT / f"{item['id']}.json"
    if output_path.exists():
        existing = json.loads(output_path.read_text())
        existing_source = existing.get('content', {}).get('source_code', '')
        if isinstance(existing_source, str) and existing_source.strip():
            print(f'[skip] source_code already exists: {output_path.name} ({len(existing_source)} characters)')
            continue
    splice = next((x['url'] for x in item['delivery'] if x['protocol'] == 'SPLICE'), None)
    try:
        if not splice: raise ValueError('no SPLICE delivery URL')
        item['content']['source_code'] = extract_source(splice)
    except Exception as exc:
        failures.append({'persistentID': old['persistentID'], 'url': splice, 'error': str(exc)})
        print('[failure]', old['persistentID'], exc)
    output_path.write_text(json.dumps(item, indent=2, ensure_ascii=False) + '\n')
(OUTPUT / 'source-code-failures.json').write_text(json.dumps(failures, indent=2) + '\n')
print(f'Wrote {len(converted)} items to {OUTPUT}; source extraction failures: {len(failures)}')


[skip] source_code already exists: cf1cbc701df0089af59da2d5.json (1400 characters)
[skip] source_code already exists: 7bbbea3b9062bcbf2b61bd2b.json (1285 characters)
[skip] source_code already exists: 4eff51a87e23f689b721d020.json (1112 characters)
[skip] source_code already exists: dbe74423d7cd56d6901799ec.json (745 characters)
[skip] source_code already exists: 70b0566b7354b5cb65cb667d.json (973 characters)
[skip] source_code already exists: 3d65ce673bd183beb6bca53a.json (3677 characters)
[skip] source_code already exists: a6aa2fde4855a8da5505de24.json (2745 characters)
[skip] source_code already exists: 9ca760672f21aca7c917f8ac.json (568 characters)
[skip] source_code already exists: b0a6f1d0d9a047e4310262ab.json (2042 characters)
[skip] source_code already exists: da08262680faaeb2025edeab.json (173 characters)
[skip] source_code already exists: 7fdd291970a20ff66af67d1a.json (534 characters)
[skip] source_code already exists: e3524550296d70aa605e8668.json (687 characters)
[skip] sou

In [43]:
required = {'id', 'identity', 'attribution', 'languages', 'content', 'classification', 'pedagogy', 'interaction', 'delivery', 'rights'}
files = sorted(p for p in OUTPUT.glob('*.json') if p.name != 'source-code-failures.json')
items = [json.loads(p.read_text()) for p in files]
ids = [x['id'] for x in items]
assert len(items) == len(converted), (len(items), len(converted))
assert len(ids) == len(set(ids)), 'Duplicate generated IDs'
assert all(required <= set(x) for x in items), 'Missing Catalog v2 fields'
assert all(len(x['id']) == 24 and all(c in '0123456789abcdef' for c in x['id']) for x in items)
assert all(x['identity']['id'] for x in items)
assert sum(x['languages']['programming_languages'] == ['Java'] for x in items) == len(json.loads(SOURCES[0].read_text()))
assert sum(x['languages']['programming_languages'] == ['C++'] for x in items) == len(json.loads(SOURCES[1].read_text()))
populated_source = sum(bool(x['content'].get('source_code', '').strip()) for x in items)
java_count = sum(x['languages']['programming_languages'] == ['Java'] for x in items)
cpp_count = sum(x['languages']['programming_languages'] == ['C++'] for x in items)
print(f'Verified {len(items)} items: {java_count} Java, {cpp_count} C++; source_code populated: {populated_source}')
if failures:
    failure_report = OUTPUT / 'source-code-failures.json'
    print(f'WARNING: {len(failures)} source-code extractions failed; review {failure_report}')


Verified 350 items: 184 Java, 166 C++; source_code populated: 350


In [44]:
PUSH_TO_SERVER = True  # Change to True only after verification

if not PUSH_TO_SERVER:
    print('Upload skipped. Set PUSH_TO_SERVER = True after reviewing catalogv2-items/.')
else:
    assert len(files) == len(converted) and not failures, 'Refusing upload: conversion or source extraction is incomplete'
    assert all(item['content']['source_code'].strip() for item in items), 'Refusing upload: one or more source_code fields are empty'
    token_path = Path(os.environ.get('CATALOG_V2_API_TOKEN_FILE', API_TOKEN_FILE))
    if not token_path.exists():
        raise FileNotFoundError(f'API token file not found: {token_path}')
    token = token_path.read_text().strip()
    for n, path in enumerate(files, 1):
        item = json.loads(path.read_text())
        item_id = item['id']
        response = requests.patch(f'{API_BASE}/slc-items-api/{item_id}', json=item, headers={'api-token': token}, timeout=60)
        response.raise_for_status()
        print(f'[{n}/{len(files)}] {response.status_code} {item_id}')


HTTPError: 404 Client Error: Not Found for url: http://adapt2.sis.pitt.edu/next.course-authoring/api/slc-items-api/0034ac7252620e98b9677a0b